In [0]:
from pyspark.sql.functions import dense_rank
from pyspark.sql.window import Window

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS marathos.gold")

In [0]:
# Läs in silver tabellen (OBT = cleaned dataset från silver_layer)
df = spark.table("marathos.silver.obt")

# Kontrollerar att alla kolumner finns i datasetet
display(df)
df.printSchema()

## Dim Event

In [0]:

# Skapa unik lista av events 
dim_event = df.select(
    "Event_name",
    "Event_distance_length"
).distinct()

# Skapa event_id
w_event = Window.orderBy("Event_name")

dim_event = dim_event.withColumn(
    "event_id",
    dense_rank().over(w_event))


# Visa resultat
display(dim_event)


In [0]:
# Spara resultatet i gold
dim_event.write.mode("overwrite").saveAsTable("marathos.gold.dim_event")

## Dim Athlete


In [0]:
# Skapa unik lista av idrottare
dim_athlete = df.select(
    "Athlete_ID",
    "Athlete_country",
    "Athlete_gender",
    "Athlete_age_category",
    ).distinct()

w_athlete = Window.orderBy("Athlete_ID")

dim_athlete = dim_athlete.withColumn("athlete_id", dense_rank().over(w_athlete))

# Visa resultatet
display(dim_athlete)

In [0]:
# Spara resultatet i gold
dim_athlete.write.mode("overwrite").saveAsTable("marathos.gold.dim_athlete")

## FCT Results

In [0]:
spark.sql("DROP TABLE IF EXISTS marathos.gold.fct_results")

In [0]:

from pyspark.sql.functions import dense_rank
from pyspark.sql.window import Window


fct_results = df.join(dim_event, ["Event_name", "Event_distance_length"], "inner") \
    .join(dim_athlete, ["Athlete_ID", "Athlete_country", "Athlete_gender", "Athlete_age_category"], "inner") \
    .select(
        dim_event["event_id"],             
        dim_athlete["athlete_id"],         
        df["Event_dates"],
        df["Athlete_average_speed"],
        df["Athlete_performance"]
    )

# Skapa ett unikt result_id
w_result = Window.orderBy(dim_event["event_id"], dim_athlete["athlete_id"])
fct_results = fct_results.withColumn("result_id", dense_rank().over(w_result))

# Spara till gold-lagret
fct_results.write.mode("overwrite").saveAsTable("marathos.gold.fct_results")

In [0]:
display(spark.table("marathos.gold.fct_results").limit(5))